In [1]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

C:\Users\Aman Chaudhary\AppData\Local\Temp\ipykernel_7648\3949787335.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
docs = (
    PyPDFLoader("./documents/book1.pdf").load() +
    PyPDFLoader("./documents/book2.pdf").load() +
    PyPDFLoader("./documents/book3.pdf").load()
)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(25, 0, 2771451258224), '/LastChar': 1, '/Widths': [833], '/BaseFont': '/NJBOIP+MathematicalPi-Three', '/FirstChar': 1, '/Encoding': IndirectObject(26, 0, 2771451258224), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 2771451258224), '/LastChar': 2, '/Widths': [778, 778], '/BaseFont': '/NJDFHF+MSAM10', '/FirstChar': 1, '/Encoding': IndirectObject(744, 0, 2771451258224), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 2771451

In [3]:
len(docs)

2123

In [ ]:
# 2) Chunk
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)

# 3) Clean text to avoid UnicodeEncodeError (surrogates from PDF extraction)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [8]:
len(chunks)

42329

In [9]:
# 3) Index (fresh collection each run)
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks, embeddings)

ResponseError: Post "http://127.0.0.1:62816/tokenize": dial tcp 127.0.0.1:62816: connectex: No connection could be made because the target machine actively refused it. (status code: 400)

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [ ]:
# 4) LLM + prompt
llm = ChatOllama(model="qwen3:1.7b", temperature=0)

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str

In [ ]:
def retrieve(state):
    q = state["question"]
    return {"docs": retriever.invoke(q)}

In [ ]:

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": out.content}


In [ ]:
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app

In [ ]:
# 5) Run
res = app.invoke({"question": "WHat is a transformer in deep learning.", "docs": [], "answer": ""})
print(res["answer"])

In [ ]:
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)